# 01 – Extract prototype
Prototype and test for the extractor. The final version lives in src/extract_headers.py

## Setup

In [1]:
import sys
print(sys.executable)
import duckdb, zstandard, pandas
print("all imports OK")

/Users/alec/lichess/.venv/bin/python3
all imports OK


## Extract function

In [2]:
import re          # pattern matching for text
import csv         # writing CSV files 
import io          # turning bytes into text
import itertools   # tools for streams (islice)
import zstandard   # installed: decompresses .zst files

INPUT_PATH = "../data/raw/lichess_db_standard_rated_2016-01.pgn.zst"

HEADER_NAMES = ["Event", "Site", "White", "Black", "Result", "UTCDate", "UTCTime",
          "WhiteElo", "BlackElo", "WhiteRatingDiff", "BlackRatingDiff",
          "ECO", "Opening", "TimeControl", "Termination", "WhiteTitle", "BlackTitle"]

# "^\[" line starts with [
# "(\w+)" group 1, key like WhiteElo
# "(.*)" group 2, value
# ""\]$" closing quote, ] at end of line
HEADER_PATTERN = re.compile(r'^\[(\w+) "(.*)"\]$') # matches a header line like [WhiteElo "1305"] 


def open_pgn_as_text(path):
    """Open a .pgn.zst file and return it as a stream of text lines."""
    file_handle = open(path, mode="rb") # "rb" = read binary
    decompressed_bytes = zstandard.ZstdDecompressor().stream_reader(file_handle)
    text_lines = io.TextIOWrapper(decompressed_bytes, encoding="utf-8") # TextIOWrapper converts bytes into text
    return text_lines

In [3]:
def extract_headers(lines, output_path):
    """Read PGN lines, write one CSV row per game. Returns the number of games written."""
    games_written = 0
    current_game = {}
    with open(output_path, mode="w", newline="", encoding="utf-8") as output_file:   # "w" = write
        csv_writer = csv.DictWriter(output_file, fieldnames=HEADER_NAMES, restval="") # restval="": if a game is missing one of the 17 headers, leave that cell empty
        csv_writer.writeheader()
        for line in lines:
            line = line.rstrip("\n")
            
            # A new [Event line means the previous game is complete, so write it and start fresh.
            if line.startswith('[Event ') and current_game: # True only when current_game has data
                csv_writer.writerow(current_game)
                games_written += 1
                current_game = {}
                    
            # store header name and value in current game
            header_match = HEADER_PATTERN.match(line) # a match object if this line is a header, otherwise None
            if header_match is not None: 
                header_name, header_value = header_match.groups()
                current_game[header_name] = header_value

        # after the loop, write the final game.
        if current_game: # True only when current_game has data
            csv_writer.writerow(current_game)
            games_written += 1
            current_game = {}

    return games_written

## Test on fake games

In [4]:
fake_lines = [
    '[Event "Game A"]', '[White "alice"]', '', ' 1-0', '',
    '[Event "Game B"]', '[White "bob"]', '', ' 0-1', '',
    '[Event "Game C"]', '[White "carol"]', '', ' 1/2-1/2',
]
print(extract_headers(fake_lines, "../data/processed/test_games.csv"))
print(open("../data/processed/test_games.csv").read())



3
Event,Site,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,WhiteTitle,BlackTitle
Game A,,alice,,,,,,,,,,,,,,
Game B,,bob,,,,,,,,,,,,,,
Game C,,carol,,,,,,,,,,,,,,



## Sample run (200,000 lines)

In [5]:
sample_lines = itertools.islice(open_pgn_as_text(INPUT_PATH), 200_000)
sample_game_count = extract_headers(sample_lines, "../data/processed/sample_games.csv")
print(sample_game_count)

11096


## Verification

In [6]:
import pandas as pd
sample_df = pd.read_csv("../data/processed/sample_games.csv", dtype=str)
print(sample_df.shape)
sample_df.head()

(11096, 17)


,Event,Site,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,WhiteTitle,BlackTitle
0,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/HoJqFz7k,t1r33,mashi,0-1,2015.12.31,23:00:03,1305,1321,-13,+14,?,?,300+0,Abandoned,NaN,NaN
1,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/FYTW1fa0,genibo77,skeith73,0-1,2015.12.31,23:00:03,1805,1628,-17,+17,?,?,300+0,Abandoned,NaN,NaN
2,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/ocl40XzA,herrmaa,Yory21,0-1,2015.12.31,23:00:03,1598,1609,-11,+12,?,?,300+0,Abandoned,NaN,NaN
3,Rated Blitz tournament https://lichess.org/tou...,https://lichess.org/exzWvUJ4,byoxz64,reyjohnny11,1-0,2015.12.31,23:00:03,1500,1792,+387,-12,B00,King's Pawn,180+0,Abandoned,NaN,NaN
4,Rated Bullet tournament https://lichess.org/to...,https://lichess.org/5d3R83G8,romanchessmaster,chunkymonkey,0-1,2015.12.31,23:00:03,1500,1477,-185,+9,C40,King's Pawn Game: McConnell Defense,60+0,Normal,NaN,NaN


In [7]:
sample_df["WhiteTitle"].value_counts(dropna=False)

WhiteTitle
NaN    10951
LM        76
NM        25
CM        17
IM        15
FM        12
Name: count, dtype: int64

In [8]:
sample_df.tail(1).T

,11095
Event,Rated Blitz game
Site,https://lichess.org/INFdYwbu
White,ban-ban
Black,JulioCesar12
Result,1-0
UTCDate,2016.01.01
UTCTime,01:20:16
WhiteElo,1676
BlackElo,1470
WhiteRatingDiff,+5


11,096 games, matching grep -c on the same 200,000 lines. Shape (11096, 17). The last game was cut off by the line limit (ECO, Opening, TimeControl, and Termination empty), which restval filled as blank; the full file ends cleanly. Title counts sum to the total row count.